In [ ]:
import geopandas as gpd
import folium as folium
from pathlib import Path

#This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
#Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)


# --- Load the GeoPackage dynamically from the output folder where it was saved by the data processing script ---
gpkg_path = output_folder / "road_condition.gpkg"
print(gpkg_path.exists(), gpkg_path) # <-- for debugging, to check if the file exists and the path is correct
#print(gdf.columns) <-- for debugging, to check the column names and ensure they match the tooltip fields

# --- Read the GeoPackage and ensure it's in WGS84 (EPSG:4326) for compatibility with web mapping ---
gdf = gpd.read_file(gpkg_path).to_crs(epsg=4326)

# --- Drop rows without geometry ---
gdf = gdf[gdf.geometry.notnull()].copy()

# --- Choose a center point for the map ---
finland_bounds = [
    [59.5, 19.0], # Southwest: latitude, longitude
    [70.5, 32.5]  # Northeast: latitude, longitude
]

#----------create the map with openstreetmap tiles and set the initial view to the center of the data---------
Map = folium.Map(
    location = [64.5, 26.0],
    zoom_start = 5,
    tiles = "OpenStreetMap",
    max_bounds = True
)

#-----limit the map to the bounds of Finland for better user experience and tiny performance boost-----
Map.fit_bounds(finland_bounds) #<-- DOES NOT WORK YET BUT DOES COMPILE. PLS DON'T TOUCH THIS FOR NOW. I WILL FIX IT LATER.


# --- Style each road using the stored color_hex field ---
def style_function(feature):
    color = feature["properties"].get("color_hex", "#808080")
    return {
        "color": color,
        "weight": 4,
        "opacity": 0.9
    }

# --- Tooltip fields ---
tooltip = folium.GeoJsonTooltip(
    fields=["Tie", "Aosa", "AET", "RMS_yhd", "color_hex"],
    aliases=["Tie", "Aosa", "AET", "RMS_yhd", "Color"],
    localize=True
)

# --- Add roads to the map ---
folium.GeoJson(
    gdf,
    style_function=style_function,
    tooltip=tooltip,
    name="Road condition"
).add_to(Map)

# --- Set map bounds to Finland ---
Map.options["min_lat"] = 59.5
Map.options["max_lat"] = 70.5
Map.options["min_lon"] = 19.0
Map.options["max_lon"] = 32.5

# --- Add layer control to toggle the road condition layer ----
# Shows on the right top corner of the map
folium.LayerControl().add_to(Map)

# --- Save to HTML ------
Map.save(output_folder / "road_condition_map.html")
print("Map saved as road_condition_map.html")